# Sports RAG - Retriever Phase Experiments

This notebook implements the retrieval phase exactly for your plan:
- BM25 on fixed chunks
- BM25 on recursive chunks
- BM25 on semantic chunks
- Hybrid (BM25 fixed + dense fixed namespace) with RRF
- Hybrid (BM25 recursive + dense recursive namespace) with RRF
- Hybrid (BM25 semantic + dense semantic namespace) with RRF

Then it evaluates all retrieval setups on 25 manual questions with Recall@5, Recall@10, and MRR@10, selects winners, and runs QA comparison on only those winners.

## 1) Install Dependencies

In [ ]:
!pip -q install rank-bm25 sentence-transformers pinecone pandas numpy tqdm transformers accelerate openai

## 2) Imports and Global Config

In [ ]:
import os
import json
import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pinecone import Pinecone
from transformers import pipeline

# Retrieval hyperparameters
TOP_K = 10
RRF_K = 60
DENSE_TOP_K = 50

# Re-ranking hyperparameters
RERANK_CANDIDATES_TOP_K = 30
QA_CONTEXT_TOP_K = 5

# LLM-as-judge configuration
JUDGE_PROVIDER = "groq"  # "groq" or "openai"
JUDGE_MODEL_NAME = "llama-3.3-70b-versatile"
JUDGE_EVAL_SIZE = 15

# Embedding model for dense retrieval queries (good quality + Kaggle-friendly size)
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

# Cross-encoder model for final re-ranking (stronger quality on Kaggle GPU)
CROSS_ENCODER_MODEL_NAME = "BAAI/bge-reranker-base"

# QA model for final winner-vs-winner answer comparison
QA_MODEL_NAME = "google/flan-t5-base"

# Kaggle Secrets support for API keys.
try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()

    pinecone_secret = user_secrets.get_secret("PINECONE_API_KEY")
    if pinecone_secret:
        os.environ["PINECONE_API_KEY"] = pinecone_secret
        print("Loaded PINECONE_API_KEY from Kaggle Secrets.")

    openai_secret = user_secrets.get_secret("OPENAI_API_KEY")
    if openai_secret:
        os.environ["OPENAI_API_KEY"] = openai_secret
        print("Loaded OPENAI_API_KEY from Kaggle Secrets.")

    groq_secret = user_secrets.get_secret("GROQ_API_KEY")
    if groq_secret:
        os.environ["GROQ_API_KEY"] = groq_secret
        print("Loaded GROQ_API_KEY from Kaggle Secrets.")
except Exception:
    # Local environment or missing Kaggle Secrets; env vars can still be set manually.
    pass

print("Imports and config loaded.")

## 3) Locate Input Files and Load Chunks

In [ ]:
def resolve_chunk_dir() -> Path:
    cwd = Path.cwd()

    # Preferred local/Kaggle working directory names
    candidate_dirs = [
        cwd / "chunks_BM25",
        cwd / "Keyword Chunks",
        cwd.parent / "chunks_BM25",
        cwd.parent / "Keyword Chunks",
    ]

    for d in candidate_dirs:
        if d.exists():
            return d

    # Kaggle dataset mount fallback: /kaggle/input/<dataset>/...
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        # First, look for a directory named chunks_BM25 anywhere under /kaggle/input
        for d in kaggle_input.rglob("chunks_BM25"):
            if d.is_dir():
                return d

        # Then, look for any directory that contains all three required files
        required = {"chunks_fixed.json", "chunks_recursive.json", "chunks_semantic.json"}
        for fixed_file in kaggle_input.rglob("chunks_fixed.json"):
            parent = fixed_file.parent
            names = {p.name for p in parent.glob("*.json")}
            if required.issubset(names):
                return parent

    raise FileNotFoundError(
        "Could not find chunk JSON folder. Expected one of: "
        "./chunks_BM25, ./Keyword Chunks, ../chunks_BM25, ../Keyword Chunks, "
        "or a Kaggle input directory containing chunks_fixed.json/chunks_recursive.json/chunks_semantic.json."
    )

CHUNK_DIR = resolve_chunk_dir()
PROJECT_ROOT = CHUNK_DIR.parent

chunk_paths = {
    "fixed": CHUNK_DIR / "chunks_fixed.json",
    "recursive": CHUNK_DIR / "chunks_recursive.json",
    "semantic": CHUNK_DIR / "chunks_semantic.json",
}

chunks_by_strategy = {}
for strategy, path in chunk_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        chunks_by_strategy[strategy] = json.load(f)

print(f"Chunk directory: {CHUNK_DIR}")
for strategy, chunks in chunks_by_strategy.items():
    print(f"{strategy}: {len(chunks)} chunks")

# Build id->chunk lookup per strategy
chunk_lookup = {
    strategy: {item["id"]: item for item in items}
    for strategy, items in chunks_by_strategy.items()
}

## 4) Build BM25 Indexes (Fixed, Recursive, Semantic)

In [ ]:
def bm25_tokenize(text: str):
    return re.findall(r"[A-Za-z0-9]+", text.lower())

bm25_indexes = {}
for strategy, chunks in chunks_by_strategy.items():
    tokenized_corpus = [bm25_tokenize(c["text"]) for c in chunks]
    bm25_indexes[strategy] = BM25Okapi(tokenized_corpus)

print("Built BM25 indexes for fixed/recursive/semantic.")

## 5) BM25 and Dense Retrieval Helpers

In [ ]:
def bm25_retrieve(query: str, strategy: str, top_k: int = TOP_K):
    chunks = chunks_by_strategy[strategy]
    bm25 = bm25_indexes[strategy]

    q_tokens = bm25_tokenize(query)
    scores = bm25.get_scores(q_tokens)
    ranked_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in ranked_idx:
        chunk = chunks[idx]
        results.append({
            "id": chunk["id"],
            "score": float(scores[idx]),
            "text": chunk.get("text", ""),
            "source": chunk.get("source", ""),
            "url": chunk.get("url", ""),
            "strategy": strategy,
        })
    return results

def init_dense_components(index_name: str = "sports-rag"):
    pinecone_api_key = os.environ.get("PINECONE_API_KEY")
    if not pinecone_api_key:
        raise EnvironmentError("Set PINECONE_API_KEY before running dense retrieval.")

    pc = Pinecone(api_key=pinecone_api_key)
    index = pc.Index(index_name)
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)
    return index, embed_model

def dense_retrieve(query: str, namespace: str, strategy: str, index, embed_model, top_k: int = TOP_K):
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()

    response = index.query(
        vector=q_vec,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
    )

    local_lookup = chunk_lookup[strategy]
    results = []

    for match in response.get("matches", []):
        match_id = match.get("id")
        metadata = match.get("metadata", {}) or {}
        fallback = local_lookup.get(match_id, {})

        results.append({
            "id": match_id,
            "score": float(match.get("score", 0.0)),
            "text": metadata.get("text", fallback.get("text", "")),
            "source": metadata.get("source", fallback.get("source", "")),
            "url": metadata.get("url", fallback.get("url", "")),
            "strategy": strategy,
        })
    return results

## 6) Reciprocal Rank Fusion (RRF) and Hybrid Retriever

In [ ]:
def reciprocal_rank_fusion(result_lists, rrf_k: int = RRF_K, top_k: int = TOP_K):
    fused_scores = defaultdict(float)
    payload = {}

    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item["id"]
            fused_scores[doc_id] += 1.0 / (rrf_k + rank)
            if doc_id not in payload:
                payload[doc_id] = item

    ranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    fused = []
    for doc_id, score in ranked:
        item = payload[doc_id].copy()
        item["score"] = float(score)
        fused.append(item)
    return fused

def hybrid_retrieve(query: str, strategy: str, namespace: str, index, embed_model, top_k: int = TOP_K):
    sparse_hits = bm25_retrieve(query, strategy=strategy, top_k=DENSE_TOP_K)
    dense_hits = dense_retrieve(
        query,
        namespace=namespace,
        strategy=strategy,
        index=index,
        embed_model=embed_model,
        top_k=DENSE_TOP_K,
    )
    return reciprocal_rank_fusion([sparse_hits, dense_hits], rrf_k=RRF_K, top_k=top_k)

## 7) Cross-Encoder Re-Ranking Helpers

In [ ]:
def init_cross_encoder(model_name: str = CROSS_ENCODER_MODEL_NAME):
    return CrossEncoder(model_name)

def cross_encoder_rerank(query: str, candidates, cross_encoder, top_k: int = TOP_K):
    if not candidates:
        return []

    # Cross-encoder scores each (query, document) pair directly.
    pairs = [[query, c.get("text", "")] for c in candidates]
    scores = cross_encoder.predict(pairs)

    reranked = []
    for item, ce_score in zip(candidates, scores):
        obj = item.copy()
        obj["rerank_score"] = float(ce_score)
        reranked.append(obj)

    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]

## 7) Sanity Check Queries

In [ ]:
sample_query = "Who won the 2022 FIFA World Cup final?"

print("BM25 fixed sample:")
for r in bm25_retrieve(sample_query, strategy="fixed", top_k=3):
    print(r["id"], "|", r["score"])

## 8) Define 25 Evaluation Questions

Fill this list with exactly 25 manually created questions.
Each item must contain:
- question: string
- relevant_chunk_ids: list of acceptable chunk IDs for relevance (from the chosen strategy files)

In [ ]:
evaluation_questions = [
    {
        "question": "What is the FIFA World Cup?",
        "relevant_chunk_ids": [
            "fixed_FIFA_World_Cup_0",
            "recursive_FIFA_World_Cup_0",
            "semantic_FIFA_World_Cup_0",
        ],
    },
    {
        "question": "Which tournament is the 2022 FIFA World Cup?",
        "relevant_chunk_ids": [
            "fixed_2022_FIFA_World_Cup_0",
            "recursive_2022_FIFA_World_Cup_0",
            "semantic_2022_FIFA_World_Cup_0",
        ],
    },
    {
        "question": "Which tournament is the 2018 FIFA World Cup?",
        "relevant_chunk_ids": [
            "fixed_2018_FIFA_World_Cup_0",
            "recursive_2018_FIFA_World_Cup_0",
            "semantic_2018_FIFA_World_Cup_0",
        ],
    },
    {
        "question": "What is FIFA in world football?",
        "relevant_chunk_ids": [
            "fixed_FIFA_0",
            "recursive_FIFA_0",
            "semantic_FIFA_0",
        ],
    },
    {
        "question": "What club is AC Milan?",
        "relevant_chunk_ids": [
            "fixed_AC_Milan_0",
            "recursive_AC_Milan_0",
            "semantic_AC_Milan_0",
        ],
    },
    {
        "question": "What club is AFC Ajax?",
        "relevant_chunk_ids": [
            "fixed_AFC_Ajax_0",
            "recursive_AFC_Ajax_0",
            "semantic_AFC_Ajax_0",
        ],
    },
    {
        "question": "What competition is the AFC Champions League Elite?",
        "relevant_chunk_ids": [
            "fixed_AFC_Champions_League_Elite_0",
            "recursive_AFC_Champions_League_Elite_0",
            "semantic_AFC_Champions_League_Elite_0",
        ],
    },
    {
        "question": "Which country does the Argentina national football team represent?",
        "relevant_chunk_ids": [
            "fixed_Argentina_national_football_team_0",
            "recursive_Argentina_national_football_team_0",
            "semantic_Argentina_national_football_team_0",
        ],
    },
    {
        "question": "What club is Arsenal F.C.?",
        "relevant_chunk_ids": [
            "fixed_Arsenal_FC_0",
            "recursive_Arsenal_FC_0",
            "semantic_Arsenal_FC_0",
        ],
    },
    {
        "question": "What is the Ballon d'Or award in football?",
        "relevant_chunk_ids": [
            "fixed_Ballon_dOr_0",
            "recursive_Ballon_dOr_0",
            "semantic_Ballon_dOr_0",
        ],
    },
    {
        "question": "What club is Borussia Dortmund?",
        "relevant_chunk_ids": [
            "fixed_Borussia_Dortmund_0",
            "recursive_Borussia_Dortmund_0",
            "semantic_Borussia_Dortmund_0",
        ],
    },
    {
        "question": "Which country does the Brazil national football team represent?",
        "relevant_chunk_ids": [
            "fixed_Brazil_national_football_team_0",
            "recursive_Brazil_national_football_team_0",
            "semantic_Brazil_national_football_team_0",
        ],
    },
    {
        "question": "What is the Bundesliga?",
        "relevant_chunk_ids": [
            "fixed_Bundesliga_0",
            "recursive_Bundesliga_0",
            "semantic_Bundesliga_0",
        ],
    },
    {
        "question": "What club is Chelsea F.C.?",
        "relevant_chunk_ids": [
            "fixed_Chelsea_FC_0",
            "recursive_Chelsea_FC_0",
            "semantic_Chelsea_FC_0",
        ],
    },
    {
        "question": "What competition is Copa Libertadores?",
        "relevant_chunk_ids": [
            "fixed_Copa_Libertadores_0",
            "recursive_Copa_Libertadores_0",
            "semantic_Copa_Libertadores_0",
        ],
    },
    {
        "question": "Who is Cristiano Ronaldo?",
        "relevant_chunk_ids": [
            "fixed_Cristiano_Ronaldo_0",
            "recursive_Cristiano_Ronaldo_0",
            "semantic_Cristiano_Ronaldo_0",
        ],
    },
    {
        "question": "Which country does the Croatia national football team represent?",
        "relevant_chunk_ids": [
            "fixed_Croatia_national_football_team_0",
            "recursive_Croatia_national_football_team_0",
            "semantic_Croatia_national_football_team_0",
        ],
    },
    {
        "question": "Who is Diego Maradona?",
        "relevant_chunk_ids": [
            "fixed_Diego_Maradona_0",
            "recursive_Diego_Maradona_0",
            "semantic_Diego_Maradona_0",
        ],
    },
    {
        "question": "What is the EFL Cup?",
        "relevant_chunk_ids": [
            "fixed_EFL_Cup_0",
            "recursive_EFL_Cup_0",
            "semantic_EFL_Cup_0",
        ],
    },
    {
        "question": "Which country does the England national football team represent?",
        "relevant_chunk_ids": [
            "fixed_England_national_football_team_0",
            "recursive_England_national_football_team_0",
            "semantic_England_national_football_team_0",
        ],
    },
    {
        "question": "What is the Eredivisie?",
        "relevant_chunk_ids": [
            "fixed_Eredivisie_0",
            "recursive_Eredivisie_0",
            "semantic_Eredivisie_0",
        ],
    },
    {
        "question": "Who is Erling Haaland?",
        "relevant_chunk_ids": [
            "fixed_Erling_Haaland_0",
            "recursive_Erling_Haaland_0",
            "semantic_Erling_Haaland_0",
        ],
    },
    {
        "question": "What is the FA Cup?",
        "relevant_chunk_ids": [
            "fixed_FA_Cup_0",
            "recursive_FA_Cup_0",
            "semantic_FA_Cup_0",
        ],
    },
    {
        "question": "What club is FC Barcelona?",
        "relevant_chunk_ids": [
            "fixed_FC_Barcelona_0",
            "recursive_FC_Barcelona_0",
            "semantic_FC_Barcelona_0",
        ],
    },
    {
        "question": "What club is FC Bayern Munich?",
        "relevant_chunk_ids": [
            "fixed_FC_Bayern_Munich_0",
            "recursive_FC_Bayern_Munich_0",
            "semantic_FC_Bayern_Munich_0",
        ],
    },
]

print("Current #questions:", len(evaluation_questions))
if len(evaluation_questions) != 25:
    print("WARNING: Add questions until you have exactly 25 for final evaluation.")
else:
    print("All 25 football-only evaluation questions loaded with real chunk IDs.")

## 9) Evaluation Metrics: Recall@5, Recall@10, MRR@10

In [ ]:
def compute_recall_at_k(pred_ids, relevant_ids, k):
    top_pred = pred_ids[:k]
    hits = len(set(top_pred).intersection(set(relevant_ids)))
    return 1.0 if hits > 0 else 0.0

def compute_mrr_at_k(pred_ids, relevant_ids, k=10):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(pred_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def evaluate_retriever(eval_data, retriever_fn, top_k=10):
    rows = []
    for item in tqdm(eval_data, desc="Evaluating"):
        q = item["question"]
        rel = item["relevant_chunk_ids"]

        results = retriever_fn(q, top_k)
        pred_ids = [r["id"] for r in results]

        rows.append({
            "question": q,
            "recall@5": compute_recall_at_k(pred_ids, rel, 5),
            "recall@10": compute_recall_at_k(pred_ids, rel, 10),
            "mrr@10": compute_mrr_at_k(pred_ids, rel, 10),
        })

    df = pd.DataFrame(rows)
    return {
        "Recall@5": df["recall@5"].mean(),
        "Recall@10": df["recall@10"].mean(),
        "MRR@10": df["mrr@10"].mean(),
        "details": df,
    }

## 10) Run All 6 Retrieval Experiments

In [ ]:
experiments = {
    "bm25_fixed": lambda q, k: bm25_retrieve(q, "fixed", top_k=k),
    "bm25_recursive": lambda q, k: bm25_retrieve(q, "recursive", top_k=k),
    "bm25_semantic": lambda q, k: bm25_retrieve(q, "semantic", top_k=k),
}

dense_ready = bool(os.environ.get("PINECONE_API_KEY"))
if dense_ready:
    index, embed_model = init_dense_components(index_name="sports-rag")
    experiments.update({
        "hybrid_fixed": lambda q, k: hybrid_retrieve(q, "fixed", "fixed", index, embed_model, top_k=k),
        "hybrid_recursive": lambda q, k: hybrid_retrieve(q, "recursive", "recursive", index, embed_model, top_k=k),
        "hybrid_semantic": lambda q, k: hybrid_retrieve(q, "semantic", "semantic", index, embed_model, top_k=k),
    })
else:
    print("PINECONE_API_KEY not set: running BM25-only experiments.")

all_results = []
for name, retriever_fn in experiments.items():
    metrics = evaluate_retriever(evaluation_questions, retriever_fn, top_k=10)
    all_results.append({
        "experiment": name,
        "Recall@5": metrics["Recall@5"],
        "Recall@10": metrics["Recall@10"],
        "MRR@10": metrics["MRR@10"],
    })

results_df = pd.DataFrame(all_results).sort_values("MRR@10", ascending=False)
results_df

## 11) Winner Selection (1 Best Sparse + 1 Best Hybrid)

In [ ]:
sparse_df = results_df[results_df["experiment"].str.startswith("bm25_")].copy()
hybrid_df = results_df[results_df["experiment"].str.startswith("hybrid_")].copy()

best_sparse = sparse_df.iloc[0]["experiment"] if not sparse_df.empty else None
best_hybrid = hybrid_df.iloc[0]["experiment"] if not hybrid_df.empty else None

print("Best sparse:", best_sparse)
print("Best hybrid:", best_hybrid)

## 12) Final QA Phase on Winners Only

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def build_context(results, max_chars=2500):
    context_parts = []
    total = 0
    for r in results:
        txt = r.get("text", "")
        if not txt:
            continue
        if total + len(txt) > max_chars:
            break
        context_parts.append(txt)
        total += len(txt)
    return "\n\n".join(context_parts)

def make_qa_prompt(question, context):
    return (
        "Answer the question using only the context. If context is insufficient, say: I don't know.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

# Load QA generator
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(QA_MODEL_NAME)
qa_device = "cuda" if torch.cuda.is_available() else "cpu"
qa_model = qa_model.to(qa_device)

# Load cross-encoder reranker for query-document scoring
cross_encoder = init_cross_encoder()

def answer_with_retriever(question, retriever_fn):
    # 1) Retrieve a larger candidate pool
    candidates = retriever_fn(question, RERANK_CANDIDATES_TOP_K)

    # 2) Re-rank with cross-encoder and keep top context chunks
    reranked_hits = cross_encoder_rerank(
        query=question,
        candidates=candidates,
        cross_encoder=cross_encoder,
        top_k=QA_CONTEXT_TOP_K,
    )

    # 3) Build QA context from reranked chunks
    context = build_context(reranked_hits)
    prompt = make_qa_prompt(question, context)

    inputs = qa_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(qa_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = qa_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )

    out = qa_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    return {
        "answer": out,
        "context": context,
        "reranked_hits": reranked_hits,
    }

winner_map = {}
for name in [best_sparse, best_hybrid]:
    if name is not None and name in experiments:
        winner_map[name] = experiments[name]

qa_rows = []
qa_artifacts = []
for item in tqdm(evaluation_questions, desc="QA Comparison (cross-encoder rerank)"):
    q = item["question"]
    row = {"question": q}

    for winner_name, retriever_fn in winner_map.items():
        result = answer_with_retriever(q, retriever_fn)
        row[f"answer_{winner_name}"] = result["answer"]

        qa_artifacts.append({
            "question": q,
            "winner": winner_name,
            "answer": result["answer"],
            "context": result["context"],
        })

    qa_rows.append(row)

qa_df = pd.DataFrame(qa_rows)
qa_artifacts_df = pd.DataFrame(qa_artifacts)
qa_df.head()

## 13) LLM-as-Judge: Faithfulness and Response Relevancy

Fixed evaluation set: first 15 queries from evaluation_questions.
Faithfulness: claim extraction and verification against retrieved context.
Response relevancy: generate 3 alternate questions from the answer and compute cosine similarity with the original query.

In [ ]:
from openai import OpenAI

def init_judge_client():
    provider = str(JUDGE_PROVIDER).strip().lower()

    if provider == "groq":
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            # Kaggle secret fetch using explicit syntax requested by user.
            try:
                from kaggle_secrets import UserSecretsClient
                user_secrets = UserSecretsClient()
                secret_value_0 = user_secrets.get_secret("GROQ_API_KEY")
                if secret_value_0:
                    api_key = secret_value_0
                    os.environ["GROQ_API_KEY"] = api_key
            except Exception:
                pass

        if not api_key:
            raise EnvironmentError(
                "GROQ_API_KEY is required for Groq judge evaluation. "
                "Set it in Kaggle Secrets with label GROQ_API_KEY."
            )
        return OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

    if provider == "openai":
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise EnvironmentError(
                "OPENAI_API_KEY is required for OpenAI judge evaluation. "
                "Set it in environment variables or Kaggle Secrets."
            )
        return OpenAI(api_key=api_key)

    raise ValueError("JUDGE_PROVIDER must be either 'groq' or 'openai'.")

def llm_json(client, system_prompt: str, user_prompt: str, model: str = JUDGE_MODEL_NAME):
    resp = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    text = resp.choices[0].message.content
    return json.loads(text)

def extract_claims(client, answer: str):
    system_prompt = "You extract factual atomic claims from answers."
    user_prompt = (
        "Extract atomic factual claims from the answer. Return JSON with key 'claims' as a list of short strings. "
        "If no factual claim is present, return an empty list.\n\n"
        f"Answer:\n{answer}"
    )
    data = llm_json(client, system_prompt, user_prompt)
    claims = data.get("claims", [])
    if not isinstance(claims, list):
        return []
    return [str(c).strip() for c in claims if str(c).strip()]

def verify_claim(client, question: str, claim: str, context: str):
    system_prompt = "You verify whether a claim is supported by provided context only."
    user_prompt = (
        "Verify if the claim is supported by the context only. "
        "Return JSON with keys: label (supported or unsupported), rationale (1-2 lines), evidence (short quote or empty).\n\n"
        f"Question: {question}\n\n"
        f"Claim: {claim}\n\n"
        f"Context:\n{context}"
    )
    data = llm_json(client, system_prompt, user_prompt)
    label = str(data.get("label", "unsupported")).strip().lower()
    if label not in {"supported", "unsupported"}:
        label = "unsupported"
    return {
        "claim": claim,
        "label": label,
        "rationale": str(data.get("rationale", "")).strip(),
        "evidence": str(data.get("evidence", "")).strip(),
    }

def generate_alternate_questions(client, answer: str):
    system_prompt = "You generate alternative user queries from an answer."
    user_prompt = (
        "Generate exactly 3 different questions that this answer could respond to. "
        "Return JSON with key 'questions' as a list of exactly 3 strings.\n\n"
        f"Answer:\n{answer}"
    )
    data = llm_json(client, system_prompt, user_prompt)
    questions = data.get("questions", [])
    if not isinstance(questions, list):
        questions = []
    questions = [str(q).strip() for q in questions if str(q).strip()]
    return questions[:3]

def cosine_similarity(a: str, b: str, embed_model):
    vecs = embed_model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(vecs[0], vecs[1]))

judge_client = init_judge_client()
relevancy_embed_model = SentenceTransformer(EMBED_MODEL_NAME)

judge_target = best_hybrid if best_hybrid in winner_map else best_sparse
if judge_target is None:
    raise ValueError("No winner retriever found for judge evaluation.")

judge_questions = [item["question"] for item in evaluation_questions[: min(JUDGE_EVAL_SIZE, len(evaluation_questions))]]
judge_input_df = qa_artifacts_df[
    (qa_artifacts_df["winner"] == judge_target) &
    (qa_artifacts_df["question"].isin(judge_questions))
].copy()

judge_rows = []
example_rows = []

for _, row in tqdm(judge_input_df.iterrows(), total=len(judge_input_df), desc="LLM-as-judge"):
    question = row["question"]
    answer = row["answer"]
    context = row["context"]

    claims = extract_claims(judge_client, answer)
    verifications = [verify_claim(judge_client, question, claim, context) for claim in claims]

    supported = sum(1 for v in verifications if v["label"] == "supported")
    total_claims = len(verifications)
    faithfulness_score = (supported / total_claims) if total_claims > 0 else 0.0

    alt_questions = generate_alternate_questions(judge_client, answer)
    alt_similarities = [cosine_similarity(question, q_alt, relevancy_embed_model) for q_alt in alt_questions]
    avg_relevancy = float(np.mean(alt_similarities)) if alt_similarities else 0.0

    judge_rows.append({
        "question": question,
        "winner": judge_target,
        "num_claims": total_claims,
        "supported_claims": supported,
        "faithfulness_score": faithfulness_score,
        "alternate_questions": json.dumps(alt_questions, ensure_ascii=False),
        "alt_question_similarities": json.dumps(alt_similarities, ensure_ascii=False),
        "average_relevancy_score": avg_relevancy,
    })

    if len(example_rows) < 3:
        example_rows.append({
            "question": question,
            "answer": answer,
            "claims": json.dumps(claims, ensure_ascii=False),
            "verification_results": json.dumps(verifications, ensure_ascii=False),
        })

judge_df = pd.DataFrame(judge_rows)
judge_examples_df = pd.DataFrame(example_rows)

print(f"Judge provider: {JUDGE_PROVIDER}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Judge target retriever: {judge_target}")
print(f"Evaluated queries: {len(judge_df)}")
print(f"Average faithfulness score: {judge_df['faithfulness_score'].mean():.4f}")
print(f"Average response relevancy score: {judge_df['average_relevancy_score'].mean():.4f}")

judge_examples_df

## 14) Ablation Study: Chunking x Retrieval Variations

This ablation compares performance across:
- Chunking: fixed, recursive, semantic
- Retrieval: semantic-only vs hybrid + cross-encoder re-ranking
- Metrics: Faithfulness and Relevancy

In [ ]:
if not dense_ready:
    raise EnvironmentError("Ablation requires dense retrieval. Set PINECONE_API_KEY and rerun retrieval cells.")

# Final run settings: top-2 variations only, full judge set, strong model
JUDGE_MODEL_NAME = "llama-3.3-70b-versatile"
ABLATION_EVAL_SIZE = min(15, len(evaluation_questions))

# Keep only the 2 best variation pairs from your pilot run.
# Format: (chunking, retrieval_variant)
ABLATION_TOP2_VARIATIONS = [
    ("fixed", "hybrid_plus_rerank"),
    ("recursive", "hybrid_plus_rerank"),
]

if "judge_client" not in globals():
    judge_client = init_judge_client()
if "relevancy_embed_model" not in globals():
    relevancy_embed_model = SentenceTransformer(EMBED_MODEL_NAME)
if "cross_encoder" not in globals():
    cross_encoder = init_cross_encoder()

def generate_answer_from_context(question: str, context: str):
    prompt = make_qa_prompt(question, context)
    inputs = qa_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    inputs = {k: v.to(qa_device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = qa_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
        )
    return qa_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

def retrieve_hits_for_ablation(question: str, chunking: str, retrieval_variant: str):
    if retrieval_variant == "semantic_only":
        return dense_retrieve(
            query=question,
            namespace=chunking,
            strategy=chunking,
            index=index,
            embed_model=embed_model,
            top_k=QA_CONTEXT_TOP_K,
        )

    if retrieval_variant == "hybrid_plus_rerank":
        hybrid_candidates = hybrid_retrieve(
            query=question,
            strategy=chunking,
            namespace=chunking,
            index=index,
            embed_model=embed_model,
            top_k=RERANK_CANDIDATES_TOP_K,
        )
        return cross_encoder_rerank(
            query=question,
            candidates=hybrid_candidates,
            cross_encoder=cross_encoder,
            top_k=QA_CONTEXT_TOP_K,
        )

    raise ValueError(f"Unknown retrieval_variant: {retrieval_variant}")

ablation_questions = [item["question"] for item in evaluation_questions[:ABLATION_EVAL_SIZE]]
ablation_rows = []

for question in tqdm(ablation_questions, desc="Ablation question loop (top-2 final run)"):
    for chunking, retrieval_variant in ABLATION_TOP2_VARIATIONS:
        selected_hits = retrieve_hits_for_ablation(question, chunking, retrieval_variant)
        context = build_context(selected_hits)
        answer = generate_answer_from_context(question, context)

        claims = extract_claims(judge_client, answer)
        verifications = [verify_claim(judge_client, question, claim, context) for claim in claims]
        supported = sum(1 for v in verifications if v["label"] == "supported")
        total_claims = len(verifications)
        faithfulness_score = (supported / total_claims) if total_claims > 0 else 0.0

        alt_questions = generate_alternate_questions(judge_client, answer)
        alt_similarities = [
            cosine_similarity(question, q_alt, relevancy_embed_model) for q_alt in alt_questions
        ]
        average_relevancy_score = float(np.mean(alt_similarities)) if alt_similarities else 0.0

        ablation_rows.append({
            "question": question,
            "chunking": chunking,
            "retrieval_variant": retrieval_variant,
            "num_claims": total_claims,
            "supported_claims": supported,
            "faithfulness_score": faithfulness_score,
            "average_relevancy_score": average_relevancy_score,
        })

ablation_detail_df = pd.DataFrame(ablation_rows)
ablation_comparison_df = (
    ablation_detail_df
    .groupby(["chunking", "retrieval_variant"], as_index=False)
    .agg(
        queries=("question", "count"),
        faithfulness=("faithfulness_score", "mean"),
        relevancy=("average_relevancy_score", "mean"),
    )
    .sort_values(["chunking", "retrieval_variant"])
    .reset_index(drop=True)
 )

print(f"Final run model: {JUDGE_MODEL_NAME}")
print(f"Final run evaluated questions: {len(ablation_questions)}")
print(f"Final run variation count: {len(ABLATION_TOP2_VARIATIONS)}")
ablation_comparison_df

## 15) Save Outputs

In [ ]:
def resolve_output_dir(project_root: Path) -> Path:
    candidates = [
        Path("/kaggle/working") / "outputs",
        Path.cwd() / "outputs",
        project_root / "outputs",
    ]

    for candidate in candidates:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            return candidate
        except OSError:
            continue

    raise OSError("Could not create a writable outputs directory.")

output_dir = resolve_output_dir(PROJECT_ROOT)

if 'results_df' in globals():
    results_df.to_csv(output_dir / "retrieval_metrics.csv", index=False)

if 'qa_df' in globals():
    qa_df.to_csv(output_dir / "qa_winner_comparison.csv", index=False)

if 'qa_artifacts_df' in globals():
    qa_artifacts_df.to_csv(output_dir / "qa_artifacts.csv", index=False)

if 'judge_df' in globals():
    judge_df.to_csv(output_dir / "llm_judge_metrics.csv", index=False)

if 'judge_examples_df' in globals():
    judge_examples_df.to_csv(output_dir / "llm_judge_examples.csv", index=False)

if 'ablation_detail_df' in globals():
    ablation_detail_df.to_csv(output_dir / "ablation_detail_metrics.csv", index=False)

if 'ablation_comparison_df' in globals():
    ablation_comparison_df.to_csv(output_dir / "ablation_comparison_metrics.csv", index=False)

print(f"Saved outputs in: {output_dir}")